# Training YOLOv8 Smart Gate

Notebook ini memakai `dataset_merged/` sebagai dataset utama. `plat-nomor/` digunakan sebagai fallback jika dataset gabungan tidak tersedia.

In [24]:
!pip install -q ultralytics pyyaml gdown

from pathlib import Path
import zipfile
import gdown
import torch

if not torch.cuda.is_available():
    raise RuntimeError('GPU CUDA tidak aktif. Di Colab pilih Runtime > Change runtime type > T4 GPU lalu jalankan ulang.')
print('GPU:', torch.cuda.get_device_name(0))

# Dataset gabungan diunduh langsung dari Google Drive.
FILE_ID = '138RROLwz9wkpmgJiwwYOl3Tf1UahDKZz'
ZIP_PATH = Path('/content/smartgate-all-datasets.zip')
if not ZIP_PATH.is_file():
    gdown.download(id=FILE_ID, output=str(ZIP_PATH), quiet=False)
if not ZIP_PATH.is_file() or ZIP_PATH.stat().st_size < 1000000:
    raise RuntimeError(f'Download ZIP gagal atau file terlalu kecil: {ZIP_PATH}')
with zipfile.ZipFile(ZIP_PATH) as archive:
    bad_member = archive.testzip()
    if bad_member is not None:
        raise RuntimeError(f'ZIP rusak pada entry: {bad_member}')
    archive.extractall('/content')

candidates = [path.parent for path in Path('/content').rglob('data.yaml') if path.parent.name in ('dataset_merged', 'plat-nomor')]
preferred = [path for path in candidates if path.name == 'dataset_merged']
DATA_ROOT = next(iter(preferred or candidates), None)
if DATA_ROOT is None:
    raise FileNotFoundError('data.yaml tidak ditemukan setelah download dan ekstraksi ZIP.')
DATA_YAML = DATA_ROOT / 'data.yaml'
print('ZIP:', ZIP_PATH, f'({ZIP_PATH.stat().st_size / 1024**3:.2f} GB)')
print('Dataset:', DATA_ROOT)
print('Config sumber:', DATA_YAML)

GPU: Tesla T4
ZIP: /content/smartgate-all-datasets.zip (2.22 GB)
Dataset: /content/dataset_merged
Config sumber: /content/dataset_merged/data.yaml


In [25]:
import yaml

config = yaml.safe_load(DATA_YAML.read_text())
config['path'] = str(DATA_ROOT)
config['train'] = 'train/images'
config['val'] = 'valid/images'
config['test'] = 'test/images'
config['nc'] = 1
config['names'] = ['number-plate']
RUNTIME_YAML = Path('/content/dataset_runtime.yaml')
RUNTIME_YAML.write_text(yaml.safe_dump(config, sort_keys=False))
print(RUNTIME_YAML.read_text())

for split in ('train', 'val', 'test'):
    image_dir = DATA_ROOT / config[split]
    label_dir = DATA_ROOT / config[split].replace('/images', '/labels')
    if not image_dir.is_dir():
        raise FileNotFoundError(f'{split} images tidak ditemukan: {image_dir}')
    if not label_dir.is_dir():
        raise FileNotFoundError(f'{split} labels tidak ditemukan: {label_dir}')
    images = {p.stem for p in image_dir.iterdir() if p.is_file()}
    labels = {p.stem for p in label_dir.glob('*.txt')}
    missing_labels = sorted(images - labels)
    orphan_labels = sorted(labels - images)
    if missing_labels or orphan_labels:
        raise ValueError(f'{split}: missing labels={len(missing_labels)}, orphan labels={len(orphan_labels)}')
    for label_path in label_dir.glob('*.txt'):
        for line_no, line in enumerate(label_path.read_text().splitlines(), 1):
            fields = line.split()
            if len(fields) != 5 or int(fields[0]) != 0 or not all(0.0 <= float(v) <= 1.0 for v in fields[1:]):
                raise ValueError(f'Anotasi invalid: {label_path}:{line_no} -> {line}')
    print(f'{split}: {len(images)} images, {len(labels)} labels, valid')
print('Struktur dan anotasi dataset valid.')

path: /content/dataset_merged
train: train/images
val: valid/images
test: test/images
nc: 1
names:
- number-plate

train: 1572 images, 1572 labels, valid
val: 290 images, 290 labels, valid
test: 84 images, 84 labels, valid
Struktur dan anotasi dataset valid.


In [26]:
def count_labels(folder):
    files = list(Path(folder).glob('*.txt'))
    objects = sum(len(path.read_text().splitlines()) for path in files)
    return len(files), objects

for split in ('train', 'valid', 'test'):
    label_dir = DATA_ROOT / split / 'labels'
    files, objects = count_labels(label_dir)
    print(f'{split}: {files} file label, {objects} anotasi')

train: 1572 file label, 2284 anotasi
valid: 290 file label, 446 anotasi
test: 84 file label, 102 anotasi


## Training

Gunakan GPU Colab: **Runtime > Change runtime type > T4 GPU**. Kotak anotasi harus tepat mengelilingi papan plat, bukan bumper, kamera, gril, atau seluruh kendaraan.

In [27]:
from ultralytics import YOLO
import torch

device = 0
batch = 16
EXPERIMENT_NAME = 'smartgate-colab-v3'
print('Device:', device)
print('Eksperimen:', EXPERIMENT_NAME)

RUN_DIR = DATA_ROOT.parent / 'runs' / 'detect'
model = YOLO('yolov8s.pt')
model.train(
    data=str(RUNTIME_YAML),
    epochs=60,
    imgsz=640,
    batch=batch,
    device=device,
    project=str(RUN_DIR),
    name=EXPERIMENT_NAME,
    patience=15,
    cos_lr=True,
    box=7.5,
    hsv_h=0.015,
    hsv_s=0.55,
    hsv_v=0.35,
    degrees=5.0,
    translate=0.08,
    scale=0.35,
    shear=0.0,
    perspective=0.0,
    fliplr=0.0,
    mosaic=0.7,
    mixup=0.05,
    close_mosaic=10,
    pretrained=True,
    plots=True,
    val=True,
    seed=42,
    workers=2,
    cache=False,
    exist_ok=True,
)
print('Training selesai.')

Device: 0
Eksperimen: smartgate-colab-v3
Ultralytics 8.4.177 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/dataset_runtime.yaml, degrees=5.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=60, erasing=0.4, exist_ok=True, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.55, hsv_v=0.35, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.05, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=0.7, multi_scale=0.0, name=smartgate-colab-v

In [28]:
best_pt = RUN_DIR / EXPERIMENT_NAME / 'weights' / 'best.pt'
if not best_pt.is_file():
    raise FileNotFoundError(best_pt)
trained = YOLO(str(best_pt))
val_metrics = trained.val(data=str(RUNTIME_YAML), split='val', imgsz=640, device=device)
test_metrics = trained.val(data=str(RUNTIME_YAML), split='test', imgsz=640, device=device)
if float(test_metrics.box.map50) < 0.98:
    print('Menjalankan evaluasi test dengan TTA...')
    test_metrics_tta = trained.val(data=str(RUNTIME_YAML), split='test', imgsz=640, device=device, augment=True)
    if float(test_metrics_tta.box.map50) > float(test_metrics.box.map50):
        test_metrics = test_metrics_tta
print('best.pt:', best_pt)
print(f'Validation mAP50: {val_metrics.box.map50:.4f}')
print(f'Validation mAP50-95: {val_metrics.box.map:.4f}')
print(f'Test mAP50: {test_metrics.box.map50:.4f}')
print(f'Test mAP50-95: {test_metrics.box.map:.4f}')

Ultralytics 8.4.177 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 72 layers, 11,125,971 parameters, 0 gradients, 28.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2321.2±1459.2 MB/s, size: 474.3 KB)
val: Scanning /content/dataset_merged/valid/labels.cache... 290 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 290/290 93.6Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 19/19 3.0it/s 6.4s0.3s
                   all        290        446      0.853       0.85      0.889      0.519
Speed: 1.8ms preprocess, 9.2ms inference, 0.0ms loss, 1.5ms postprocess per image
Results saved to /content/runs/detect/val-7
Ultralytics 8.4.177 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 72 layers, 11,125,971 parameters, 0 gradients, 28.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 69.3±37.3 MB/s, size: 231.8 KB)
val: Scanning /

In [29]:
print('Bobot selesai:', best_pt)
print('File model tersedia:', best_pt.is_file())

Bobot selesai: /content/runs/detect/smartgate-colab-v3/weights/best.pt
File model tersedia: True


In [30]:
!pip install -q easyocr opencv-python-headless onnx onnxruntime

import cv2
import re
import time
import json
import numpy as np
import easyocr

ocr_reader = easyocr.Reader(['en'], gpu=torch.cuda.is_available())
ALLOWLIST = 'ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789'
DIGIT_TO_CHAR = {'0': 'O', '1': 'I', '2': 'Z', '3': 'B', '4': 'A', '5': 'S', '6': 'G', '7': 'T', '8': 'B', '9': 'Q'}
CHAR_TO_DIGIT = {'O': '0', 'D': '0', 'Q': '0', 'I': '1', 'L': '1', 'Z': '2', 'A': '4', 'S': '5', 'G': '6', 'T': '7', 'B': '8'}
VALID_PREFIXES = {'A','B','D','E','F','G','H','K','L','M','N','P','R','S','T','W','Z','AA','AB','AD','AE','AG','BA','BB','BD','BE','BG','BH','BK','BL','BM','BN','BP','DA','DB','DC','DD','DE','DG','DH','DK','DL','DM','DN','DP','DR','DT','DW','EA','EB','ED','KB','KH','KT','KU','PA','PB'}

def assess_crop_quality(crop_bgr):
    gray = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2GRAY)
    blur_score = float(cv2.Laplacian(gray, cv2.CV_64F).var())
    brightness = float(np.mean(gray))
    return {'blur_score': round(blur_score, 2), 'brightness': round(brightness, 2), 'is_usable': blur_score >= 30.0 and 20.0 <= brightness <= 245.0}

def detect_plate_color(crop_bgr):
    hsv = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2HSV)
    h,w = hsv.shape[:2]
    center = hsv[int(h*.20):int(h*.75), int(w*.15):int(w*.85)]
    total = center.shape[0] * center.shape[1] + 1e-5
    yellow = cv2.inRange(center, (15,80,80), (35,255,255))
    red = cv2.bitwise_or(cv2.inRange(center,(0,80,80),(10,255,255)), cv2.inRange(center,(160,80,80),(180,255,255)))
    if np.sum(yellow > 0) / total > .25: return 'YELLOW'
    if np.sum(red > 0) / total > .25: return 'RED'
    return 'WHITE' if float(np.mean(center[:,:,2])) > 127 else 'BLACK'

def preprocess_plate_for_ocr(crop_bgr, plate_color):
    h,w = crop_bgr.shape[:2]
    x_trim = int(w * .03)
    y_top = int(h * .04)
    y_bot = max(y_top + 10, int(h * .70))
    zone = crop_bgr[y_top:y_bot, x_trim:max(x_trim + 10, w-x_trim)]
    mh,mw = zone.shape[:2]
    if mw < 260:
        zone = cv2.resize(zone, None, fx=260.0/max(mw,1), fy=260.0/max(mw,1), interpolation=cv2.INTER_CUBIC)
    gray = cv2.cvtColor(zone, cv2.COLOR_BGR2GRAY)
    if plate_color == 'BLACK' or float(np.mean(gray)) < 110: gray = cv2.bitwise_not(gray)
    enhanced = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8)).apply(gray)
    return cv2.copyMakeBorder(enhanced, 6, 6, 8, 8, cv2.BORDER_REPLICATE)

def extract_main_ocr_text(ocr_out, img_h):
    if not ocr_out: return '', 0.0
    info=[]
    for bbox,text,conf in ocr_out:
        ys=[pt[1] for pt in bbox]; xs=[pt[0] for pt in bbox]
        clean=re.sub(r'[^A-Z0-9]','',text.upper())
        if clean: info.append({'x_min':min(xs),'y_center':float(np.mean(ys)),'height':max(ys)-min(ys),'text':clean,'conf':float(conf)})
    if not info: return '', 0.0
    max_h=max(item['height'] for item in info)
    main=[item for item in info if item['height'] >= .45*max_h and item['y_center'] <= .82*img_h] or info
    main.sort(key=lambda item:item['x_min'])
    return ''.join(item['text'] for item in main), float(np.mean([item['conf'] for item in main]))

def format_plate(raw_text):
    clean=re.sub(r'[^A-Z0-9]','',raw_text.upper())
    if len(clean) < 3: return clean, False
    if len(clean) >= 6 and clean[0] in ('I','1','L','0','O'):
        possible=''.join(DIGIT_TO_CHAR.get(c,c) for c in clean[1:3])
        if possible in VALID_PREFIXES: clean=clean[1:]
    best=None; best_score=-999
    for start in range(min(2,len(clean))):
        for length in range(min(9,len(clean)-start),2,-1):
            sub=clean[start:start+length]
            for p_len in (2,1):
                if len(sub) <= p_len: continue
                for d_len in range(min(4,len(sub)-p_len),0,-1):
                    s_len=len(sub)-p_len-d_len
                    if s_len < 0 or s_len > 3: continue
                    p_raw=sub[:p_len]; d_raw=sub[p_len:p_len+d_len]; s_raw=sub[p_len+d_len:]
                    p=''.join(DIGIT_TO_CHAR.get(c,c) for c in p_raw)
                    d=''.join(CHAR_TO_DIGIT.get(c,c) for c in d_raw)
                    s=''.join(DIGIT_TO_CHAR.get(c,c) for c in s_raw)
                    if not (p.isalpha() and d.isdigit() and (not s or s.isalpha())) or d[0]=='0' or p not in VALID_PREFIXES: continue
                    score=(sum(4 for a,b in zip(p_raw,p) if a==b)+sum(5 for a,b in zip(d_raw,d) if a==b)+sum(4 for a,b in zip(s_raw,s) if a==b)-start*8)
                    if 2 <= d_len <= 4 and 1 <= s_len <= 3: score += 12
                    if score > best_score: best_score=score; best=f'{p} {d} {s}'.strip()
    return (best, True) if best is not None and best_score >= 30 else (clean, False)

def read_smartgate_plate(image_bgr, yolo_model, conf_thres=.35):
    started=time.perf_counter(); result=yolo_model.predict(image_bgr,conf=conf_thres,imgsz=640,verbose=False)[0]; detections=[]
    h_img,w_img=image_bgr.shape[:2]
    for box in result.boxes:
        x1,y1,x2,y2=map(int,box.xyxy[0].tolist()); det_conf=float(box.conf[0])
        crop=image_bgr[max(0,y1):min(h_img,y2),max(0,x1):min(w_img,x2)]
        if crop.size==0: continue
        quality=assess_crop_quality(crop); color=detect_plate_color(crop); processed=preprocess_plate_for_ocr(crop,color)
        ocr_out=ocr_reader.readtext(processed,allowlist=ALLOWLIST,detail=1,paragraph=False)
        raw,ocr_conf=extract_main_ocr_text(ocr_out,processed.shape[0]); plate,valid=format_plate(raw)
        detections.append({'bbox':[x1,y1,x2,y2],'plate_number':plate,'raw_ocr':raw,'is_valid_format':valid,'plate_color':color,'det_conf':round(det_conf,4),'ocr_conf':round(ocr_conf,4),'quality':quality})
    return {'latency_ms':round((time.perf_counter()-started)*1000,2),'plates':detections}

test_image_dir=DATA_ROOT/'test'/'images'; test_images=sorted(p for p in test_image_dir.iterdir() if p.is_file()); latencies=[]; valid_reads=0; detected=0; sample_outputs=[]
for img_path in test_images:
    image=cv2.imread(str(img_path))
    if image is None: continue
    result=read_smartgate_plate(image,trained); latencies.append(result['latency_ms']); detected += len(result['plates']); valid_reads += sum(1 for item in result['plates'] if item['is_valid_format'])
    if len(sample_outputs)<10 and result['plates']: sample_outputs.append({'file':img_path.name,**result['plates'][0]})
onnx_path=trained.export(format='onnx',imgsz=640,simplify=True,dynamic=False)
test_map50=float(test_metrics.box.map50)
report={'model':str(best_pt),'onnx_model':str(onnx_path),'validation':{'mAP50':float(val_metrics.box.map50),'mAP50_95':float(val_metrics.box.map)},'test':{'images':len(test_images),'mAP50':test_map50,'mAP50_95':float(test_metrics.box.map)},'ocr_and_gate_benchmark':{'total_detected_plates':detected,'valid_indonesian_format_count':valid_reads,'valid_format_rate':round(valid_reads/max(detected,1),4),'avg_end_to_end_latency_ms':round(float(np.mean(latencies)),2) if latencies else 0.0,'p95_end_to_end_latency_ms':round(float(np.percentile(latencies,95)),2) if latencies else 0.0},'detector_target_mAP50':.98,'detector_target_met':test_map50>=.98,'production_kpi_status':'NOT_CLAIMED','sample_reads':sample_outputs}
REPORT_PATH=RUN_DIR/EXPERIMENT_NAME/'evaluation_summary.json'
REPORT_PATH.write_text(json.dumps(report,indent=2))
print('=== LAPORAN EVALUASI DETECTOR & OCR SMART GATE ===')
print(json.dumps(report,indent=2))

Ultralytics 8.4.177 🚀 Python-3.13.15 torch-2.11.0+cu130 CPU (Intel Xeon CPU @ 2.00GHz)
Model summary (fused): 72 layers, 11,125,971 parameters, 0 gradients, 28.4 GFLOPs

PyTorch: starting from '/content/runs/detect/smartgate-colab-v3/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 5, 8400) (21.5 MB)

ONNX: starting export with onnx 1.23.2 opset 18...
ONNX: slimming with onnxslim 0.1.98...
ONNX: export success ✅ 2.1s, saved as '/content/runs/detect/smartgate-colab-v3/weights/best.onnx' (42.7 MB)

Export complete (2.5s)
Results saved to /content/runs/detect/smartgate-colab-v3/weights/best.onnx
Predict:         yolo predict task=detect model=/content/runs/detect/smartgate-colab-v3/weights/best.onnx imgsz=640 
Validate:        yolo val task=detect model=/content/runs/detect/smartgate-colab-v3/weights/best.onnx imgsz=640 data=/content/dataset_runtime.yaml  
Visualize:       https://netron.app
=== LAPORAN EVALUASI DETECTOR & OCR SMART GATE ===
{
  "model": "/c

In [31]:
import os
import shutil
from IPython.display import FileLink, display

ARTIFACT_DIR = RUN_DIR / EXPERIMENT_NAME
required_files = [best_pt, REPORT_PATH]
missing = [str(path) for path in required_files if not path.is_file()]
if missing:
    raise FileNotFoundError('Artefak belum tersedia: ' + ', '.join(missing))

DOWNLOAD_DIR = Path('/content/smartgate-artifacts')
DOWNLOAD_DIR.mkdir(exist_ok=True)
shutil.copy2(best_pt, DOWNLOAD_DIR / 'best.pt')
shutil.copy2(REPORT_PATH, DOWNLOAD_DIR / 'evaluation_summary.json')
for result_name in ('results.csv', 'confusion_matrix.png', 'PR_curve.png', 'F1_curve.png', 'best.onnx'):
    result_path = (Path(onnx_path) if result_name == 'best.onnx' else ARTIFACT_DIR / result_name)
    if result_path.is_file():
        shutil.copy2(result_path, DOWNLOAD_DIR / result_name)

# Simpan di working directory kernel agar dapat diambil oleh VS Code/Jupyter.
ARCHIVE_PATH = Path.cwd() / 'smartgate-artifacts.zip'
if ARCHIVE_PATH.exists():
    ARCHIVE_PATH.unlink()
shutil.make_archive(str(ARCHIVE_PATH.with_suffix('')), 'zip', DOWNLOAD_DIR)
if not ARCHIVE_PATH.is_file() or ARCHIVE_PATH.stat().st_size == 0:
    raise RuntimeError(f'ZIP artefak gagal dibuat: {ARCHIVE_PATH}')
print('Artefak siap diunduh:', ARCHIVE_PATH)
print(f'Ukuran ZIP: {ARCHIVE_PATH.stat().st_size / 1024**2:.2f} MB')
print('File ZIP lokal kernel:', ARCHIVE_PATH)
display(FileLink(ARCHIVE_PATH.name, result_html_prefix='Download artefak: '))

# Runtime Colab dari VS Code tidak dapat mengirim FileLink langsung ke Windows.
# Salin artefak ke Google Drive agar dapat diambil dari drive.google.com.
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    DRIVE_ARTIFACT_DIR = Path('/content/drive/MyDrive/SmartGateSystem')
    DRIVE_ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
    DRIVE_ARCHIVE_PATH = DRIVE_ARTIFACT_DIR / ARCHIVE_PATH.name
    shutil.copy2(ARCHIVE_PATH, DRIVE_ARCHIVE_PATH)
    if not DRIVE_ARCHIVE_PATH.is_file() or DRIVE_ARCHIVE_PATH.stat().st_size != ARCHIVE_PATH.stat().st_size:
        raise IOError(f'Verifikasi salinan Drive gagal: {DRIVE_ARCHIVE_PATH}')
    print('Artefak tersalin ke Google Drive:', DRIVE_ARCHIVE_PATH)
    print('Buka drive.google.com > My Drive > SmartGateSystem untuk mengambil ZIP.')
except (ImportError, OSError, IOError) as exc:
    print('Salinan Google Drive gagal:', exc)

# Download JavaScript hanya dipanggil pada web Google Colab.
if os.environ.get('COLAB_RELEASE_TAG'):
    from google.colab import files
    files.download(str(ARCHIVE_PATH))
else:
    print('Mode VS Code/Jupyter terdeteksi; gunakan tautan FileLink di atas.')

Artefak siap diunduh: /content/smartgate-artifacts.zip
Ukuran ZIP: 55.78 MB
File ZIP lokal kernel: /content/smartgate-artifacts.zip


/content/smartgate-artifacts.zip

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Artefak tersalin ke Google Drive: /content/drive/MyDrive/SmartGateSystem/smartgate-artifacts.zip
Buka drive.google.com > My Drive > SmartGateSystem untuk mengambil ZIP.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Setelah training selesai, salin `best.pt` ke:

`Tugas Citra Digital/prototype/runs/detect/smartgate-colab-v2/weights/best.pt`

Lalu jalankan aplikasi dengan `SGS_YOLO_WEIGHTS` menunjuk ke file tersebut dan restart Flask.